In [1]:
import torch
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [2]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        # Em classificação multiclasse com CrossEntropyLoss, os rótulos precisam ser inteiros (long)
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        return self._x[idx], self._y[idx]

In [3]:
dataset = load_iris()
features = dataset.data
targets = dataset.target

In [4]:
max_epochs = 500
batch = 8
learning_rate = 0.001
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
def train_model_class(features, targets, kf, batch, epochs, lr):
    acc_scores = []
    f1_scores = []
    
    for i, (train_index, test_index) in enumerate(kf.split(features)):
        print(f"\nTreinando modelo ----- Fold {i + 1}")
        
        # 1. Instanciando o modelo do zero para este Fold
        model = Sequential(
            Linear(4, 32),
            ReLU(),
            Linear(32, 16),
            ReLU(),
            Linear(16, 3)  # 3 saídas para as 3 classes do Iris
        )
        
        optmizer = Adam(model.parameters(), lr=lr)
        criterion = CrossEntropyLoss()
        
        # 2. Criando o Dataset e DataLoader com as amostras de treino do Fold atual
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch, shuffle=True)
        
        # 3. Loop de Treinamento
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        # 4. Loop de Avaliação/Teste do Fold atual
        model.eval()
        with torch.no_grad():
            x_test_tensor = torch.from_numpy(features[test_index]).float()
            y_true = targets[test_index]
            
            # Passagem do conjunto de teste pelo modelo
            output = model(x_test_tensor)
            
            # Pega o índice com maior valor em cada previsão (classe predita)
            result = torch.argmax(output, dim=1).cpu().numpy()
            
            # Cálculo das métricas
            acc_s = accuracy_score(y_true, result)
            f1_s = f1_score(y_true, result, average="macro")
            cm = confusion_matrix(y_true, result)
            
            acc_scores.append(acc_s)
            f1_scores.append(f1_s)
            
            print(f"Acurácia: {acc_s:.4f}")
            print(f"F1-Score: {f1_s:.4f}")
            print(f"Matriz de Confusão:\n{cm}")

    # 5. Resultado final combinado de todos os Folds
    print("\n==========================================")
    print(f"Resultado Médio das Dobras ----- Acurácia: {np.mean(acc_scores):.4f}, F1-Score: {np.mean(f1_scores):.4f}")

In [6]:
train_model_class(features, targets, kf, batch, max_epochs, learning_rate)


Treinando modelo ----- Fold 1
Acurácia: 1.0000
F1-Score: 1.0000
Matriz de Confusão:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Treinando modelo ----- Fold 2
Acurácia: 0.9667
F1-Score: 0.9585
Matriz de Confusão:
[[13  0  0]
 [ 0 10  0]
 [ 0  1  6]]

Treinando modelo ----- Fold 3
Acurácia: 0.9667
F1-Score: 0.9628
Matriz de Confusão:
[[12  0  0]
 [ 0  9  1]
 [ 0  0  8]]

Treinando modelo ----- Fold 4
Acurácia: 0.9333
F1-Score: 0.9373
Matriz de Confusão:
[[ 8  0  0]
 [ 0  8  2]
 [ 0  0 12]]

Treinando modelo ----- Fold 5
Acurácia: 1.0000
F1-Score: 1.0000
Matriz de Confusão:
[[ 7  0  0]
 [ 0 11  0]
 [ 0  0 12]]

Resultado Médio das Dobras ----- Acurácia: 0.9733, F1-Score: 0.9717
